<a href="https://colab.research.google.com/github/hrijutadph21/DFEN-DR-Classification/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np

# 1. Green Channel Extraction
def extract_green_channel(image):
    return image[:, :, 1]

# 2. Resize Image
def resize_image(image, size=(224, 224)):
    return cv2.resize(image, size)

# 3. CLAHE (Contrast Enhancement)
def apply_clahe(image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(image)

# 4. Normalization
def normalize_image(image):
    image = image / 255.0
    return image

# 5. Full Preprocessing Pipeline
def preprocess_image(image_path):
    image = cv2.imread(image_path)

    # Resize
    image = resize_image(image)

    # Extract Green Channel
    green = extract_green_channel(image)

    # Apply CLAHE
    enhanced = apply_clahe(green)

    # Normalize
    normalized = normalize_image(enhanced)

    # Expand dims for CNN input
    normalized = np.expand_dims(normalized, axis=-1)

    return normalized



from tensorflow.keras.applications import (
    EfficientNetB3, MobileNetV2, ResNet50,
    VGG19, DenseNet121
)
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

def build_model(base_model):
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(5, activation='softmax')(x)  # 5 DR classes
    return Model(inputs=base_model.input, outputs=x)

def get_models(input_shape=(224,224,3)):
    models = []

    base_models = [
        EfficientNetB3(weights='imagenet', include_top=False, input_shape=input_shape),
        MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape),
        ResNet50(weights='imagenet', include_top=False, input_shape=input_shape),
        VGG19(weights='imagenet', include_top=False, input_shape=input_shape),
        DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    ]

    for base in base_models:
        models.append(build_model(base))

    return models

from tensorflow.keras.applications import (
    EfficientNetB3, MobileNetV2, ResNet50,
    VGG19, DenseNet121
)
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model
import tensorflow as tf

NUM_CLASSES = 5

# -------------------------------
# Build Base Model
# -------------------------------
def build_base_model(base_model):
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    return Model(inputs=base_model.input, outputs=x)

# -------------------------------
# Load All Models
# -------------------------------
def get_base_models(input_shape=(224,224,3)):
    base_models = [
        EfficientNetB3(weights='imagenet', include_top=False, input_shape=input_shape),
        MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape),
        ResNet50(weights='imagenet', include_top=False, input_shape=input_shape),
        VGG19(weights='imagenet', include_top=False, input_shape=input_shape),
        DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    ]

    models = [build_base_model(m) for m in base_models]
    return models

# -------------------------------
# 1. Feature-Level Fusion
# -------------------------------
def feature_fusion_model(input_shape=(224,224,3)):
    inputs = Input(shape=input_shape)

    base_models = get_base_models(input_shape)

    features = []
    for model in base_models:
        features.append(model(inputs))

    # Concatenate feature vectors
    fused = Concatenate()(features)

    # Meta-classifier (MLP)
    x = Dense(256, activation='relu')(fused)
    x = Dense(128, activation='relu')(x)
    mlp_output = Dense(NUM_CLASSES, activation='softmax', name="mlp_output")(x)

    return Model(inputs=inputs, outputs=mlp_output)

# -------------------------------
# 2. Weighted Averaging Fusion
# -------------------------------
def weighted_average(predictions, weights):
    weights = tf.constant(weights, dtype=tf.float32)
    weights = weights / tf.reduce_sum(weights)  # Normalize (Eq.1)

    weighted_preds = [w * p for w, p in zip(weights, predictions)]
    ensemble = tf.add_n(weighted_preds)  # Eq.2

    return ensemble  # final softmax vector

# -------------------------------
# 3. Soft Fusion (Final DFEN)
# -------------------------------
def soft_fusion(mlp_output, weighted_output, alpha=0.5):
    final_score = alpha * mlp_output + (1 - alpha) * weighted_output  # Eq.4
    return final_score

import os
from preprocessing import preprocess_image
import numpy as np

def load_data(folder):
    data = []
    labels = []

    for label in os.listdir(folder):
        class_path = os.path.join(folder, label)
        for img in os.listdir(class_path):
            img_path = os.path.join(class_path, img)
            processed = preprocess_image(img_path)
            data.append(processed)
            labels.append(int(label))

    return np.array(data), np.array(labels)

if __name__ == "__main__":
    X, y = load_data("data/")
    print("Data shape:", X.shape)